In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda3/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
import pandas as pd

In [4]:
def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

In [8]:
X_fp_RU_pooled = np.load("Vectors/X_fp_pooled_RU.npy")
y=np.load("Vectors/y.npy")

In [6]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [10]:
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import DataStructs

gen = GetMorganGenerator(radius=2, fpSize=1024, includeChirality=False)

def poly_fp(smiles_field):
    # copolymers have '.'-joined components; combine their fps (union) or use whole string
    mol = Chem.MolFromSmiles(str(smiles_field))
    if mol is None:
        return None
    return gen.GetFingerprint(mol)

fps = [poly_fp(s) for s in df["repeat_units"]]
n_bad = sum(f is None for f in fps)
print(f"{n_bad} unparseable" )
# for Butina we need distances among the valid ones — handle None if any

0 unparseable


In [11]:
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
import numpy as np

n = len(fps)   # the clean per-polymer fingerprints you just built (407, 0 unparseable)

# pairwise distances (1 - Tanimoto), lower triangle, for Butina
dist = []
for i in range(n):
    if i:
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dist.extend(1 - s for s in sims)

In [15]:
from rdkit.ML.Cluster import Butina
import numpy as np, json
from rdkit import DataStructs

clusters_04 = Butina.ClusterData(dist, n, 0.4, isDistData=True)
print(f"{len(clusters_04)} clusters at 0.4")

# reassign whole clusters to ~15-18% test
rng = np.random.default_rng(7)
clist = [list(c) for c in clusters_04]; rng.shuffle(clist)
cap = int(0.18 * n)
test_rows = []
for c in clist:
    if len(test_rows) + len(c) <= cap:
        test_rows.extend(c)
test_idx_scaffold = np.array(sorted(test_rows))
train_idx_scaffold = np.array(sorted(set(range(n)) - set(test_rows)))

# max-sim distribution
train_r = list(train_idx_scaffold)
sims = np.array([max(DataStructs.BulkTanimotoSimilarity(fps[i], [fps[j] for j in train_r]))
                 for i in test_idx_scaffold])
print(f"train={len(train_idx_scaffold)} test={len(test_idx_scaffold)} ({len(test_idx_scaffold)/n:.1%})")
print(f"max-sim: min={sims.min():.2f} median={np.median(sims):.2f} max={sims.max():.2f}")
print(f"  test with sim<0.3: {(sims<0.3).sum()}   sim<0.5: {(sims<0.5).sum()}")

54 clusters at 0.4
train=334 test=73 (17.9%)
max-sim: min=0.36 median=0.56 max=0.85
  test with sim<0.3: 0   sim<0.5: 28


In [17]:
from rdkit import Chem
from collections import defaultdict

canon = [Chem.MolToSmiles(Chem.MolFromSmiles(str(s))) for s in df["repeat_units"]]
test_set = set(test_idx_scaffold.tolist())
chem_rows = defaultdict(list)
for i, c in enumerate(canon):
    chem_rows[c].append(i)
leaked = [c for c, rows in chem_rows.items()
          if any(r in test_set for r in rows) and any(r not in test_set for r in rows)]
print(f"leakage: {len(leaked)} chemistries span both sides")   # want 0

leakage: 0 chemistries span both sides


In [18]:
import json, numpy as np
from sklearn.model_selection import KFold

def save_split(path, train_idx, test_idx, split_type, n_folds=5, seed=42, extra=None):
    train_idx = np.asarray(train_idx)
    test_idx  = np.asarray(test_idx)
    # 5 folds within trainval, ABSOLUTE indices (matching original schema)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = [{"train": train_idx[tr].tolist(), "val": train_idx[va].tolist()}
             for tr, va in kf.split(train_idx)]
    d = {
        "test":  test_idx.tolist(),
        "folds": folds,
        "meta": {"split_type": split_type,
                 "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
                 **(extra or {})},
    }
    with open(path, "w") as f:
        json.dump(d, f, indent=2)
    print(f"saved {path}: test={len(test_idx)} train={len(train_idx)} folds={len(folds)} [{split_type}]")

In [19]:
save_split("scaffold_splits_reg.json", train_idx_scaffold, test_idx_scaffold,
           "scaffold_butina_0.4", extra={"leakage": 0, "n_clusters": len(clusters_04),
                                          "median_maxsim_to_train": 0.56, "min_maxsim": 0.36})
# re-verify from disk
s = json.load(open("scaffold_splits_reg.json"))
test_set = set(s["test"])
print("saved. leakage:", len([c for c,r in chem_rows.items()
      if any(x in test_set for x in r) and any(x not in test_set for x in r)]))  # 0

saved scaffold_splits_reg.json: test=73 train=334 folds=5 [scaffold_butina_0.4]
saved. leakage: 0


In [20]:
with open("scaffold_splits_reg.json") as f: s = json.load(f)
test_set = set(s["test"])
leaked = [c for c, rows in chem_rows.items()
          if any(r in test_set for r in rows) and any(r not in test_set for r in rows)]
print("scaffold file leakage:", len(leaked))                    # 0
print("train∩test overlap:", len(test_set & set(s.get("train", []))))  # 0
print("train+test == 407:", len(s["test"]) + len(json.load(open("scaffold_splits_reg.json"))["meta"]["n_train"]) if False else len(s["test"]))

scaffold file leakage: 0
train∩test overlap: 0
train+test == 407: 73


In [21]:
with open("scaffold_splits_reg.json") as f: s = json.load(f)
all_val = set()
for fold in s["folds"]:
    all_val |= set(fold["val"])
train_set = set(range(407)) - set(s["test"])
print("val union == trainval:", all_val == train_set)          # True
print("no test in any fold:", len(set(s["test"]) & all_val))   # 0

val union == trainval: True
no test in any fold: 0
